# Νευρωνικά Δίκτυα: Feedforward Neural Networks

Σε αυτό το Colab Notebook θα μάθουμε πως χτίζουμε νευρωνικά δίκτυα με πολλαπλά επίπεδα, και με μη-γραμμικές συναρτήσεις, με την βιβλιοθήκη **Pytorch**.

Πιο συγκεκριμένα, θα φτιάξουμε:

1. Ένα απλό νευρωνικό δίκτυο με μόνο ένα επίπεδο/layer χωρίς ReLU
2. Ένα νευρωνικό δίκτυο με παραπάνω από ένα επίπεδο, πάλι χωρίς ReLU.
3. Ένα νευρωνικό δίκτυο με 2 επίπεδα και με ReLU.

Αφού δούμε πως χτίζουμε τα νευρωνικά δίκτυα, θα δούμε και πως λειτουργούν.

* Πρώτα, θέλουμε να δούμε τις λεπτομέρειες του "feedforward" -- αυτό απλά σημαίνει να δούμε πως κάνουνε τον υπολογισμό από το δεδομένα εισόδου, μέχρι το τελικό ``output`` που μας δίνουν. Αυτό αντιστοιχεί στην εντολή ``model.predict(X)`` που έχουμε ήδη δεί.

* Σαν άσκηση, σας αφήνουμε να βρείτε παραμέτρους για ένα μικρό νευρωνικό δίκτυο που υπολογίζει την συνάρτηση ``XOR``.

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```


In [2]:
# import the PyTorch library -- open-source machine learning library
# for building and training deep learning models.
# used for tasks in computer vision, natural language processing, etc.

import torch
import torch.nn as nn
import torch.nn.functional as F


## Δέντρα απόφασης

Πρίν αρχίσουμε, ας ξαναθυμηθούμε τα Δέντα Απόφασης.

Όταν δίνουμε την εντολή:
```
mymodel = DecisionTree(max_depth = 1)
```
ορίζουμε μία οικογένεια αλγορίθμων.

Αυτή η οικογένεια έχει 4 παραμέτρους:
1. ποιό feature θα χρησιμοποιήσουμε για τον διαχωρισμό
2. σε ποιό επίπεδο θα γίνει ο διαχωρισμός: BMI > α
3. εάν όντως το BMI > α, το ``output`` είναι β
4. εάν έχουμε BMI < α, το ``output`` είναι γ

Οπότε για να μπορούμε να χρησιμοποιήσουμε το δέντρο απόφασης για πρόβλεψη, πρέπει να δώσουμε/διαλέξουμε/βρούμε τιμές στις παραμέτρους.

Αυτό το κάναμε χρησιμοποιώντας τα δεδομένα εκπαίδευσης (και την εντολή ``mymodel.fit(X,y)``).

Μόλις διαλέξουμε (βρούμε) τιμές για τις παραμέτρους, τότε μπορούμε να προβλέψουμε τιμές εξόδου (labels) με την εντολή:

```
mymodel.predict(x)
```

Θα κάνουμε κάτι παρόμοιο με νευρωνικά δίκτυα. Μόνο που πρέπει **μόνοι μας** να δημιουργήσουμε την οικογένεια που αντιστοιχεί στο ```DecisionTreeClassifier(max_depth = 1)``` που την βρήκαμε έτοιμη από την βιβλιοθήκη ``sklearn``.

## Ορίζουμε το Νευρωνικό Δίκτυο

* Την οικογένεια την ονομάζουμε "VerySimpleNet".

* Μπορούμε να φανταστούμε την οικογένεια "VerySimpleNet" σε αναλογία με, για παράδειγμα, την οικογένεια των δέντρων αποφάσεων με κάποιο συγκεκριμένο βάθος.

* Η οικογένεια VerySimpleNet περιέχει άπειρα νευρωνικά δίκτυα, αλλά όλα έχουνε την ίδια μορφή -- απλά με διαφορετικές παραμέτρους. Παρομοίως, η οικογένεια δέντρων αποφάσεων με βάθος τρία: DecisionTree(max_depth = 3), περιέχει όλα (τα άπειρα) δέντρα απόφασης που έχουν βάθος 3.



### Πρώτο παράδειγμα

Στις παρακάτω εντολές βλέπουμε πως χτίζουμε τα νευρωνικά δίκτυα.
1. Στο πρώτο ``def`` ορίζουμε τα επίπεδα (εδώ έχουμε μόνο ένα)
```
self.fc1 = nn.Linear(3, 1)
```
Αυτή η εντολή ορίζει ένα επίπεδο που παίρνει τρείς αριθμούς: $x_1, x_2, x_3$, τους πολαπλασιάζει με τρείς παραμέτρους, $\alpha_1, \alpha_2, \alpha_3$, προσθέτη μία παράμετρος $\beta$, και εν τέλη επιστρέφη
$$
\alpha_1 x_1 + \alpha_2 x_2 + \alpha_3 x_3 + \beta.
$$

Σε αυτό το στάδιο, οι παράμετροι $(\alpha_1, \alpha_2, \alpha_3, \beta)$ δεν έχουν κάποιες συγκεκριμένες τιμές, ακριβώς όπως και όταν ορίζουμε ένα δέντρο απόφασης.

2. Στο δεύτερο ``def`` ο κώδικας συνδέει τα επίπεδα που ορίσαμε στο πρώτο ``def``, περιγράφοντας ακριβώς την δομή του νευρωνικού δικτύου. Σε αυτό το πρώτο παράδειγμα, υπάρχει μόνο ένα επίπεδο. Ο κώδικδας
```
x = self.fc1(x)
return x
```
λέει πως τα δεδομένα περνάνε από το επίπεδο ``self.fc1()`` και αυτό που βγάζει είναι το output.



**Σημείωση.** Κάποιες λεπτομέρειες του κώδικα έχουν να κάνουν με αντικειμενοστραφή προγραμματισμό (object oriented programming) και δεν χρειάζεται να το καταλάβουμε λεπτομερώς εδώ. Συγκεκριμένα, Θα εστιάσουμε μόνο στο κομμάτι του κώδικα που είναι σημαντικό για τους σκοπούς μας, και που θα τροποποιούμε για τα νευρωνικά δίκτυα που θα χτίσουμε. Οι γραμμές
```
class VerySimpleNet(nn.Module):
    def __init__(self):
        super(VerySimpleNet, self).__init__()
```
είναι ίδιες σε όλα τα νευρωνικά που θα δούμε.

In [3]:
# Define the neural network
class VerySimpleNet(nn.Module): #the class inherits from the nn.Module
    def __init__(self): # initialize the network layers.
        super(VerySimpleNet, self).__init__() #calls the parent class's constructor to properly initialize the nn.Module.
        self.fc1 = nn.Linear(3, 1)  # a fully connected (linear) layer with 3 inputs and 1 output

    def forward(self, x): # defines how the data passes through the network.
        x = self.fc1(x) # it is passed through self.fc1
        return x



### Δεύτερο παράδειγμα

```
def __init__(self):
      super(LessSimpleNet, self).__init__()
      self.fc1 = nn.Linear(4, 3)  
      self.fc2 = nn.Linear(3, 1)  

def forward(self, x):
      x = self.fc1(x)
      x = self.fc2(x)
      return x
```

* Το πρώτο ``def`` ορίζει δύο επίπεδα.
```
self.fc1 = nn.Linear(4, 3)
```
Αυτό το επίπεδο παίρνει τέσσερεις αριθμούς, $x_1, x_2, x_3, x_4$ και επιστρέφει τρείς:
$$
h_1 = \alpha_{11}x_1 + \alpha_{12}x_2 + \alpha_{13}x_3 + \alpha_{14}x_4 + \beta_1
$$
$$
h_2 = \alpha_{21}x_1 + \alpha_{22}x_2 + \alpha_{23}x_3 + \alpha_{24}x_4 + \beta_2
$$
$$
h_3 = \alpha_{31}x_1 + \alpha_{32}x_2 + \alpha_{33}x_3 + \alpha_{34}x_4 + \beta_3
$$
```
self.fc2 = nn.Linear(3, 1)
```
Αυτό το επίπεδο παίρνει τρείς αριθμούς, $h_1, h_2, h_3$ και επιστρέφει έναν:
$$
v = \gamma_1 h_1 + \gamma_2 h_2 + \gamma_3 h_3 + \delta.
$$
Θυμίζουμε πάλι πως όλες οι παράμετροι δεν έχουν ακόμα κάποιες συγκεκριμένες αξίες.

* Το δεύτερο ``def`` μας λέει πως συνδέονται τα δύο επίπεδα:
```
x = self.fc1(x)
x = self.fc2(x)
return x
```
Το $x$ περνάει πρώτα από το ``self.fc1()`` και βγάζει ένα καινούργιο διάνυσμα που πάλι το ονομάζουμε ``x``, και μετά αυτό περνάει από το ``self.fc2()``. Αυτό που βγάζει το ``self.fc2()`` είναι το τελικό output.


In [4]:
# Define the neural network
class LessSimpleNet(nn.Module):
    def __init__(self):
        super(LessSimpleNet, self).__init__() #calls the parent class's constructor to properly initialize the nn.Module.
        self.fc1 = nn.Linear(4, 3)  # a fully connected (linear) layer with 3 inputs and 1 output
        self.fc2 = nn.Linear(3, 1)  # a fully connected (linear) layer with 3 inputs and 1 output

    def forward(self, x): #defines how the data passes through the network.
        x = self.fc1(x) #it is passed through self.fc1
        x = self.fc2(x) #it is passed through self.fc2
        return x


## Ορίζουμε μία οικογένεια νευρωνικών δικτύων

Τώρα έχουμε δημιουργήσει δύο οικογένεις νευρωνικών δικτύων: ``VerySimpleNet()`` και ``LessSimpleNet()`` που αντιστοιχούν στην εντολή που ορίζει ένα δέντρο απόφασης, που βρήκαμε έτοιμη στην βιβλιοθήκη sklearn:
```
mymodel = DecisionTreeClassifier(max_depth=3)
```
Τώρα μπορούμε να δώσουμε την εντολή:
```
mymodel = VerySimpleNet()
```
ή
```
mymodel = LessSimpleNet()
```




### Τι κάνει αυτή η οικογένεια;

Όταν δόσουμε την εντολή
```
mymodel = DecisionTreeClassifier(max_depth=3)
```
για να χρησιμοποιήσουμε το ``mymodel`` για εκτίμηση, πρέπει οι παράμετροι του δέντρου να πάρουν συγκεκριμένες αξίες (π.χ. μέσω της εκπαίδευσης πάνω σε δεδομένα, $(X,y)$). Μόλις γίνει αυτό, τότε χρησιμοποιούμε το δέντρο για εκτίμηση με την εντολή:
```
mymodel.predict(x)
```

Κάτι πολύ παρόμοιο ισχύει και με τα νευρωνικά δίκτυα στην Pytorch.

Όταν δώσουμε την εντολή
```
mymodel = VerySimpleNet()
```
έχουμε δημιουργήσει ένα δίκτυο που παίρνει σαν input τρείς αριθμούς $x_1$, $x_2$, $x_3$. Το output του δικτύου ορίζεται μόνο όταν δώσουμε συγκεκριμένες τιμές στις παραμέτρους $\alpha_1, \alpha_2, \alpha_3$ και $\beta$. Μόλις γίνει αυτό (μέσω εκπαίδευσης, ή, όπως θα κάνουμε παρακάτω μέσω δικιά μας επιλογής) η εντολή που αντιστοικεί στην εντολή ``model.predict(x)`` είναι
```
mymodel(x)
```
Με αυτήν την εντολή, εφόσον το $x$ είναι το διάνυσμα $(x_1,x_2,x_3)$, έχουμε:

$$
{\rm mymodel}(x) = \alpha_1x_1 + \alpha_2 x_2 + \alpha_3 x_3 + \beta.
$$

Ας τα δούμε όλα αυτά τώρα στην πράξη.

In [5]:
# Αυτή η εντολή αντιστοιχεί στην εντολή mymodel = DecisionTree(max_depth = k)
mymodel = VerySimpleNet()

### Επιλέγουμε ένα συγκεκριμένο δίκτυο

Από την οικογένεια VerySimpleNet, επιλέγουμε ένα συγκεκριμένο δίκτυο, ορίζοντας συγκεκριμένες τιμές για τις παραμέτρους: α1, α2, α3, και β. Απλά για το παράδειγμά μας, επιλέγουμε
$$
\alpha_1 = 1
$$
$$
\alpha_2 = 1
$$
$$
\alpha 3 = 2
$$
$$
\beta = 2
$$


In [6]:
# Manually set weights and biases for the network
mymodel.fc1.weight = nn.Parameter(torch.tensor([[1., 1., 2.]])) # τιμές για το α1, α2, α3
mymodel.fc1.bias = nn.Parameter(torch.tensor([2.])) # τιμή για το β


In [7]:
mymodel

VerySimpleNet(
  (fc1): Linear(in_features=3, out_features=1, bias=True)
)

### Τι Κάνει το Δίκτυό μας;

Θα δούμε τι κάνει, δηλαδή τι output βγάζει, όταν του δώσουμε σαν input:

* πρώτο σημείο: (1,2,3)
* δεύτερο σημείο: (4,5,6)
* τρίτο σημείο: (7,8,9)


In [8]:
# Create a small dataset of integer inputs
inputs = torch.tensor([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]], dtype=torch.float32)

# Feed inputs to the network and print outputs
outputs = mymodel(inputs)
print(outputs)

tensor([[11.],
        [23.],
        [35.]], grad_fn=<AddmmBackward0>)


### Ο υπολογισμός

Τι ακριβώς έκανε το δίκτυο; Ας δούμε ένα από τα τρία παραδείγματα. Έχουμε σαν input:
$$
x = (1,2,3)
$$
Με τις τιμές για το $(α_1,α_2,α_3)$ και $β$ που διαλεξαμε, έχουμε:
$$
{\rm mymodel}(x) = \alpha_1 * 1 + \alpha_2 * 2  + \alpha_3 * 3 + \beta = 1 * 1 + 1 * 2 + 2 * 3 + 2 = 11.
$$


## Ένα Πιο Βαθύ Μη Γραμμικό Δίκτυο

Ορίζουμε τώρα μια οικογένεια νευρωνικών δικτύων με δύο επίπεδα, και με μία μη-γραμμική συνάρτηση. Πάλι τονίζουμε πως δεν έχουμε ορίσει ένα συγκεκριμένο δίκτυο, αλλά *οικογένεια δικτύων*, μια που δεν έχουν κάποιες συγκεκριμένες τιμές οι παράμετροι.

(Ακριβώς όπως συμβαίνει με την οικογένεια δέντρων απόφασης κάποιου βάθους.)

### Χτίζουμε το δίκτυο

Όπως και με τα πρώτα δύο παραδείγματα:

1. Στο πρώτο ```def``` ορίζουμε τα επίπεδα.
```
self.fc1 = nn.Linear(3, 2)
self.fc2 = nn.Linear(2, 1)output
self.relu = nn.ReLU()
```
2. Στο δεύτερο ```def``` τα χρησιμοποιούμε σαν τουβλάκια για να χτίσουμε το δίκτυό μας.
```
x = self.fc1(x)
x = self.relu(x)
x = self.fc2(x)
return x
```
* Πρώτα περνάει από το ```self.fc1``` -- το πρώτο γραμμικό επίπεδο.
* Ύστερα περνάει από την μη-γραμμική συνάρτιση "ReLU"
$$
{\rm ReLU}(α) = \max(0,α)
$$
Για παράδειγμα:
  * ReLU(3) = 3
  * ReLU(-1) = 0
* Τελικά, περνάει από το δεύτερο γραμμικό επίπεδο, το ```self.fc2```.

In [9]:
# Define the neural network
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(3, 2)  # 3 inputs to 2 outputs
        self.fc2 = nn.Linear(2, 1)  # 2 inputs to 1 output
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x



## Γιατί χρειαζόμαστε τα ReLU;

Τα ReLU μας επιτρέπουμε να χτίζουμε νευρωνικά δίκτυα με πολύ περίπλοκη σχέση από τα input στα output.

Γιατί τα χρειαζόμαστε; Ας χτίσουμε ένα νευρωνικό δίκτυο με δύο επίπεδα, **αλλά χωρίς ReLU**:

* input: $(x_1,x_2)$

* πρώτο επίπεδο:
$$
{\rm fc1}(x_1,x_2) = (α*x_1 + β*x_2 + c_1, γ*x_1 + δ*x_2 + c_2)
$$

* δεύτερο επίπεδο:
$$
{\rm fc2}(z_1,z_2) = μ*z_1 + ν*z_2 + c_3
$$

Τώρα όταν τα βάλουμε μαζί, έχουμε:

$$
{\rm fc2}({\rm fc1}(x_1,x_2)) = fc2(α*x_1 + β*x_2 + c_1, γ*x_1 + δ*x_2 + c_2) = μ*(αx_1+βx_2 + c_1) + ν*(γ*x_1+δ*x_2+c_2) + c_3
$$

Ξαναγράφωντας, βλέπουμε πως αυτό μας δίνει:
$$
μ*(αx_1+βx_2 + c_1) + ν*(γ*x_1+δ*x_2+c_2) + c_3 = (μα + νγ)*x_1 + (μβ + νδ) x_2 + (μc_1 + νc_2 + c_3)
$$
που έχει την μορφή ενός απλού γραμμικού επιπέδου:

$$
{\rm fc2}({\rm fc1}(x_1,x_2)) = (μα + νγ)*x_1 + (μβ + νδ) x_2 + (μc_1 + νc_2 + c_3) = A*x_1 + B*x_2 + C.
$$

In [10]:
# Create an instance of the network
model = SimpleNet()

## Τι κάνει αυτή η οικογένεια;

* input: $(x_1,x_2,x_3)$

* πρώτο επίπεδο:
$$
{\rm fc1}(x_1,x_2,x_3) = (α*x_1 + β*x_2 + γ*x_3 + c_1, ζ*x_1 + η*x_2 + θ*x_3 + c_2) = (v_1,v_2)
$$
* ReLU:
$$
ReLU(v_1,v_2) = (\max(v_1,0),\max(v_2,0)) = (z_1,z_2)
$$
* δεύτερο επίπεδο:
$$
{\rm fc2}(z_1,z_2) = μ*z_1 + ν*z_2 + c_3
$$

Συνολικά, το νευρωνικό ``SimpleNet`` έχει παράμετρους: $α, β, γ, c1; ζ, η, θ, c_2; μ, ν, c_3$.

Παίρνει σαν input τρείς αριθμούς $x_1$, $x_2$, $x_3$, και μας δίνει:

* Input: $x_1, x_2, x_3$

* output 1ου επιπέδου: $[α*x_1 + β*x_2 + γ*x_3 + c_1, ζ*x_1 + η*x_2 + θ*x_3 + c_2]$

* output 2ου επιπέδου είναι $(z_1,z_2) =$  ReLU(output1)

* output 3ου επιπέδου είναι: $μ*z_1 + ν*z_2 + c_3$

**Για αυτούς που ξέρουνε για πίνακες για διανύσματα**: Εάν γράψουμε
$$
W_1 = [[α, β, γ]; [ζ, η, θ]], \,\, {\bf c} = [c_1;c2]
$$
$$
W_2 = [μ,ν]
$$
Τότε μπορούμε να γράψουμε πιο απλά: ${\bf x} = (x_1,x_2,x_3)$,
$$
{\rm mymodel}(x_1,x_2,x_3) = W_2({\rm ReLU}(W_1*{\bf x} + {\bf c})) + c_3
$$


**Ας τα δούμε τώρα όλα αυτά στην πράξη.** Επιλέγουμε τιμές για τις παράμετρους:
* α, β, γ = 1, 0, 3
* ζ, η, θ = -1, 2, 1
* c1, c2 = 1, 2
* μ, ν = 1, 2
* c3 = 1

In [11]:
# Manually set weights and biases for the network
model.fc1.weight = nn.Parameter(torch.tensor([[1., 0., 3.], [-1., 2., 1.]])) # α, β, ...
model.fc1.bias = nn.Parameter(torch.tensor([0., 1.])) # c1, c2
model.fc2.weight = nn.Parameter(torch.tensor([[1., 2.]])) # μ, ν
model.fc2.bias = nn.Parameter(torch.tensor([1.])) # c3



In [12]:
model

SimpleNet(
  (fc1): Linear(in_features=3, out_features=2, bias=True)
  (fc2): Linear(in_features=2, out_features=1, bias=True)
  (relu): ReLU()
)

### Τι κάνει το ``SimpleNet``

Τι output θα βγάλει με input = (1,2,3);
* ${\rm self.fc1}(1,2,3) = (10,7)
$
* $
{\rm ReLU}(10,7) = (10,7)
$
* $
{\rm self.fc2}(10,7) = 25.
$

Δοκιμάστε και μόνοι σας να υπολογίσετε το output με input (4, 5, 6) και (7, 8, 9).

In [13]:
# Create a small dataset of integer inputs
inputs = torch.tensor([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]], dtype=torch.float32)

# Feed inputs to the network and print outputs
outputs = model(inputs)
print(outputs)


tensor([[25.],
        [49.],
        [73.]], grad_fn=<AddmmBackward0>)


# Puzzle: XOR_NNet()

Προσπαθήστε να βρείτε παραμέτρους για το παρακάτω νευρωνικό δίκτυο, ώστε να έχει την συμπεριφορά του XOR.

In [14]:
# Define the neural network
class XOR_NNet(nn.Module):
    def __init__(self):
        super(XOR_NNet, self).__init__()
        self.fc1 = nn.Linear(2, 2)
        self.fc2 = nn.Linear(2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [15]:
# τώρα ορίζουμε την οικογένεια
mymodel = XOR_NNet()

### Επιλέξτε παραμέτρους

Διαλέξτε τιμές για τις παραμέτρους α, β, γ, δ, c1, c2, μ, ν, c3, για να φτιάξετε το XOR.

In [16]:
# Uncomment and manually set weights and biases for the network

model.fc1.weight = nn.Parameter(torch.tensor([[-1., -1.], [2., 2.]]))
model.fc1.bias = nn.Parameter(torch.tensor([0., -1.]))
model.fc2.weight = nn.Parameter(torch.tensor([[1., 1.]]))
model.fc2.bias = nn.Parameter(torch.tensor([2.]))

In [17]:
# Φτιάχνουμε τα inputs
inputs = torch.tensor([[0,0],
                       [0,1],
                       [1,0],
                       [1,1]], dtype=torch.float32)


Εάν διαλέξετε τις παραμέτρους σωστά, τo ``XOR_NNet`` θα πρέπει να βγάζει σαν output: 0, 1, 1, 0. Το δικό σας τι output βγάζει;

In [18]:
# Feed inputs to the network and print outputs
outputs = model(inputs)
print(outputs)

tensor([[2.],
        [3.],
        [3.],
        [5.]], grad_fn=<AddmmBackward0>)
